In [1]:
#Apply PCA
import pandas as pd
from sklearn.preprocessing import minmax_scale, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.cluster import Birch
from sklearn.cluster import OPTICS
import numpy as np
import time

In [ ]:
#Load Data
df= pd.read_csv("data/fashion_mnist_full.csv")
X = df.drop(columns=["label"], errors="ignore")
print("Features shape scaled version):", X.shape)

Features shape scaled version): (70000, 784)


In [3]:
scaler = MinMaxScaler(feature_range=(0, 1))
X_sp = scaler.fit_transform(X)
print("Features shape (scaled version):", X_sp.shape)

Features shape (scaled version): (70000, 784)


In [4]:
#apply PCA
pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_sp)#Apply PCA
#print("PCA features shape:", X_pca.shape)
print("PCA-reduced features shape:", X_pca.shape)
print("Explained variance ratio sum:", sum(pca.explained_variance_ratio_))

PCA-reduced features shape: (70000, 188)
Explained variance ratio sum: 0.9502053074144728


In [5]:
#Define Clustering Parameters
k_values = range(2, 9)  # clusters for KMeans, GMM, Agglomerative, Spectral
n_init = 10              # random initialization
dbscan_eps = [0.5, 1.0, 1.5]  # DBSCAN eps values
min_samples = 5

#Function to Compute Metrics
def compute_metrics(X_data, labels):
    sil = silhouette_score(X_data, labels)
    db = davies_bouldin_score(X_data, labels)
    ch = calinski_harabasz_score(X_data, labels)
    return sil, db, ch

In [ ]:
#K-Means on Scaled + PCA Data
start_time = time.time()
kmean_pca = []
for k in k_values:
    km = KMeans(n_clusters=k, n_init=n_init, random_state=42)
    labels = km.fit_predict(X_pca)
    sil, db, ch = compute_metrics(X_pca, labels)
    kmean_pca.append({"algorithm": "KMeans", "preprocessing": "PCA", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
    
end_time = time.time()
runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"K-Means runtime: {runtime:.4f} seconds")

Runtime: 989.1953132152557 seconds
K-Means runtime: 989.1953 seconds


In [11]:
start_time = time.time()
#Gaussian Mixture (GMM)on Scaled + PCA Data
gmm_pca = []
for k in k_values:
    gmm = GaussianMixture(n_components=k, n_init=n_init, random_state=42)
    labels = gmm.fit_predict(X_pca)
    sil, db, ch = compute_metrics(X_pca, labels)
    gmm_pca.append({"algorithm": "GMM", "preprocessing": "PCA", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
    


end_time = time.time()

runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"GMM runtime: {runtime:.4f} seconds")

c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\mixture\_base.py:275: ConvergenceWarning: Best performing initialization did not converge. Try different init parameters, or increase max_iter, tol, or check for degenerate data.
  warnings.warn(
c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\mixture\_base.py:275: ConvergenceWarning: Best performing initialization did not converge. Try different init parameters, or increase max_iter, tol, or check for degenerate data.
  warnings.warn(
c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\mixture\_base.py:275: ConvergenceWarning: Best performing initialization did not converge. Try different init parameters, or increase max_iter, tol, or check for degenerate data.
  warnings.warn(
c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\skle

Runtime: 14406.99021267891 seconds
GMM runtime: 14406.9902 seconds


In [15]:
start_time = time.time()
#Agglomerative Clustering on Scaled + PCA Data
agg_pca = []
for k in k_values:
    agg = AgglomerativeClustering(n_clusters=k, linkage="ward")
    #labels = agg.fit_predict(X_pca)
    #sil, db, ch = compute_metrics(X_pca, labels)
    def compute_metrics_sampled(X, labels, sample_size=5000):
            if len(X) > sample_size:
                idx = np.random.choice(len(X), sample_size, replace=False)
                X_sample = X[idx]
                labels_sample = labels[idx]
            else:
                X_sample = X
                labels_sample = labels

            return compute_metrics(X_sample, labels_sample)
    agg_pca.append({"algorithm": "Agglomerative", "preprocessing": "PCA", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
    


end_time = time.time()

runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"Agglomerative runtime: {runtime:.4f} seconds")

Runtime: 0.0 seconds
Agglomerative runtime: 0.0000 seconds


In [17]:
start_time = time.time()
#Spectral Clustering on Scaled + PCA Data
spec_pca = []
for k in k_values:
    spec = SpectralClustering(n_clusters=k, affinity="nearest_neighbors")
    labels = spec.fit_predict(X_pca)
    sil, db, ch = compute_metrics(X_pca, labels)
    spec_pca.append({"algorithm": "Spectral", "preprocessing": "PCA", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
    


end_time = time.time()

runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"Spectral runtime: {runtime:.4f} seconds")

Runtime: 7021.903762817383 seconds
Spectral runtime: 7021.9038 seconds


In [ ]:
start_time = time.time()
#DBSCAN on Scaled + PCA Data
dbscan_pca = []
for eps in dbscan_eps:
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    labels = dbscan.fit_predict(X_pca)
    
    # Remove noise points (-1)
    mask = labels != -1
    if np.sum(mask) > 1 and len(set(labels[mask])) > 1:  # silhouette requires >= 2 points
        #sil, db, ch = compute_metrics(X_pca[mask], labels[mask])
        dbscan_pca.append({"algorithm": "DBSCAN", "preprocessing": "PCA", "eps": eps, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()

runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"DBSCAN runtime: {runtime:.4f} seconds")

Runtime: 97.19994974136353 seconds
DBSCAN runtime: 97.1999 seconds


In [22]:
start_time = time.time()
optics_pca = []
min_samples_values = [3, 5, 10, 20]

for m in min_samples_values:
    optics = OPTICS(min_samples=m, xi=0.05, n_jobs=-1)
    labels = optics.fit_predict(X_pca)

    # Remove noise points (-1) if needed
    unique_labels = set(labels) - {-1}

    if len(unique_labels) > 1:
        sil, db, ch = compute_metrics(X_pca, labels)
        optics_pca.append({
            "algorithm": "OPTICS",
            "preprocessing": "PCA",
            "min_samples": m,
            "xi": 0.05,
            "n_clusters": len(unique_labels),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })



end_time = time.time()

runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"Optics runtime: {runtime:.4f} seconds")

Runtime: 16291.81841635704 seconds
Optics runtime: 16291.8184 seconds


In [9]:
start_time = time.time()
birch_pca = []
threshold_values = [0.2, 0.5, 1.0, 1.5]

for t in threshold_values:
    birch = Birch(n_clusters=None, threshold=t)
    labels = birch.fit_predict(X_pca)

    if len(set(labels)) > 1:
        #sil, db, ch = compute_metrics(X_pca, labels)
        def compute_metrics_sampled(X, labels, sample_size=5000):
            if len(X) > sample_size:
                idx = np.random.choice(len(X), sample_size, replace=False)
                X_sample = X[idx]
                labels_sample = labels[idx]
            else:
                X_sample = X
                labels_sample = labels

            return compute_metrics(X_sample, labels_sample)
        birch_pca.append({
            "algorithm": "BIRCH",
            "preprocessing": "PCA",
            "threshold": t,
            "n_clusters": len(set(labels)),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })


end_time = time.time()

runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"Birch runtime: {runtime:.4f} seconds")

Runtime: 159.90486192703247 seconds
Birch runtime: 159.9049 seconds


In [ ]:
import csv

ames_results_pca = (kmean_pca + gmm_pca + agg_pca + spec_pca + dbscan_pca+birch_pca+optics_pca)

# Desired column order
keys = ["algorithm", "preprocessing", "k", "silhouette", "davies_bouldin", "calinski_harabasz", "eps"]

with open('updated_data/ames_data/ames_pca.csv', 'w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=keys, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(ames_results_pca)
